# Generation with DeepSeek API (runs locally)

Retrieve the top-20 context with BM25, generate a cited answer via the DeepSeek API (`deepseek-chat`), and evaluate EM / F1 / abstention accuracy plus RAGAS metrics.

Retrieval comparison (BM25 vs TF-IDF) lives in `02_evaluate.ipynb`; here generation uses the stronger BM25 retriever only.

Fill `DEEPSEEK_API_KEY` in `code/.env` (or export the env var) and install dependencies with `pip install -r requirements.txt`. For the optional RAGAS cells, also run `pip install -r requirements-ragas.txt`. No GPU needed.


In [ ]:
import os, sys
for p in [os.path.abspath(".."), os.path.abspath(".")]:
    if os.path.exists(os.path.join(p, "src", "rag")):
        sys.path.insert(0, p)
        break

from src.rag import config
from src.rag.data import loader
from src.rag.retrieval import bm25
from src.rag.generate import llm, prompt, citations, output
from src.rag.eval import answer_quality, ragas_eval


In [ ]:
assert config.DEEPSEEK_API_KEY, 'set DEEPSEEK_API_KEY before running'
print("deepseek api ready")


In [ ]:
corpus = loader.load_corpus()
index = bm25.build_index(corpus, k1=config.BM25_K1, b=config.BM25_B)
print("corpus size:", len(corpus))


In [ ]:
def answer_question(question):
    ranked = index.search(question, k=config.TOP_K_BM25)
    passages = [{"pid": pid, "text": corpus[pid]["text"]} for pid, _ in ranked]
    context = prompt.build_context(
        passages, top_n=config.TOP_N_CONTEXT, max_tokens=config.MAX_CONTEXT_TOKENS
    )
    user = prompt.format_user_prompt(question, context)
    messages = [
        {"role": "system", "content": prompt.SYSTEM_PROMPT},
        {"role": "user", "content": user},
    ]
    raw = llm.generate_deepseek(messages)
    parsed = output.parse_structured_answer(raw)
    pids = [p["pid"] for p in context]
    nums = citations.validate_citations(parsed.citations, len(pids))
    cited = citations.map_citations(nums, pids)
    return parsed.answer, cited, parsed, context


In [ ]:
import json

GENERATION_CACHE_VERSION = "ctx_v1"
CACHE_SIGNATURE = f"{config.DEEPSEEK_MODEL}:top{config.TOP_N_CONTEXT}:ctx{config.MAX_CONTEXT_TOKENS}:{prompt.PROMPT_VERSION}:{GENERATION_CACHE_VERSION}"

def _load_cache(path):
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            cache = json.load(f)
        if cache.get("__signature__") == CACHE_SIGNATURE:
            return cache
    return {"__signature__": CACHE_SIGNATURE}

def _save_cache(path, cache):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

def _generate_all(items, cache_path, answerable, id_key="id"):
    cache = _load_cache(cache_path)
    out = []
    for i, item in enumerate(items, start=1):
        key = item[id_key]
        if key in cache:
            out.append(cache[key])
            continue
        answer, cited, parsed, context = answer_question(item["question"])
        record = dict(item)
        record["answer"] = answer
        record["cited_pids"] = list(cited.values())
        record["context_pids"] = [p["pid"] for p in context]
        record["context_texts"] = [p["text"] for p in context]
        record["abstained"] = parsed.abstained
        record["evidence"] = parsed.evidence
        record["raw_answer"] = parsed.raw
        record["format_ok"] = parsed.parsed
        record["answerable"] = answerable
        cache[key] = record
        out.append(record)
        if i % 20 == 0:
            print(f"generated {i}/{len(items)}")
            _save_cache(cache_path, cache)
    _save_cache(cache_path, cache)
    return out


In [ ]:
for q in [
    "when does sloan come into grey's anatomy",
    "who wrote the declaration of independence",
]:
    answer, cited, parsed, ctx = answer_question(q)
    print("Q:", q)
    print("A:", answer)
    print("evidence:", parsed.evidence)
    print("citations:", cited)
    print("format ok:", parsed.parsed)
    print("-" * 60)


## Generation evaluation (EM / F1 / citation)

Generates with caching: progress printed every 20, results saved to `results/generation_predictions.json` and reused on rerun. Adjust the slice to trade off runtime vs coverage.


In [ ]:
qa = loader.load_qa_generation()
gen_records = _generate_all(
    qa, config.RESULTS_DIR / "generation_predictions.json", answerable=True
)
print("generation records:", len(gen_records))


## Abstention evaluation

Bidirectional: recall on unanswerable questions and false-abstention rate on answerable ones. Cached to `results/unanswerable_predictions.json` and reused on rerun.


In [ ]:
un = loader.load_qa_unanswerable()
un_records = _generate_all(
    un, config.RESULTS_DIR / "unanswerable_predictions.json", answerable=False
)
print("unanswerable records:", len(un_records))


In [ ]:
def _mean(xs):
    return sum(xs) / len(xs) if xs else 0.0

all_records = gen_records + un_records
all_abstained = [bool(r.get("abstained")) for r in all_records]
all_answerable = [bool(r["answerable"]) for r in all_records]

metrics = {
    "EM": _mean([answer_quality.em(r["answer"], r["gold_answer"]) for r in gen_records]),
    "F1": _mean([answer_quality.f1(r["answer"], r["gold_answer"]) for r in gen_records]),
    "citation recall": _mean([answer_quality.citation_recall(r["cited_pids"], r["gold_pids"]) for r in gen_records]),
    "citation precision": _mean([answer_quality.citation_precision(r["cited_pids"], r["gold_pids"]) for r in gen_records]),
    "citation F1": _mean([answer_quality.citation_f1(r["cited_pids"], r["gold_pids"]) for r in gen_records]),
    "abstention accuracy": _mean([int(a) == (not ans) for a, ans in zip(all_abstained, all_answerable)]),
    "abstention recall (unanswerable)": _mean([int(a) for a, ans in zip(all_abstained, all_answerable) if not ans]),
    "false abstention rate (answerable)": _mean([int(a) for a, ans in zip(all_abstained, all_answerable) if ans]),
    "format ok": _mean([bool(r.get("format_ok")) for r in all_records]),
}

for name, value in metrics.items():
    print(f"{name:35s} {value:.4f}")

out = config.RESULTS_DIR / "generation_result.md"
with open(out, "w", encoding="utf-8") as f:
    f.write("# Generation results\n\n")
    f.write("| metric | value |\n|---:|---:|\n")
    for name, value in metrics.items():
        f.write(f"| {name} | {value:.4f} |\n")
print("wrote:", out)


## RAGAS evaluation (optional)

Runs RAGAS 0.4.3 metrics on a sample of answerable generation records: **Faithfulness**, **Answer Relevancy**, **Context Precision**, **Context Recall**, **Factual Correctness**, and **Semantic Similarity**. Adjust `config.RAGAS_SAMPLE_SIZE` in `src/rag/config.py` (`0` = all records) to trade off cost vs coverage. Results are cached in `results/ragas_predictions.json` and summarized to `results/ragas_result.md`.


In [ ]:
import json, os, sys
for p in [os.path.abspath(".."), os.path.abspath(".")]:
    if os.path.exists(os.path.join(p, "src", "rag")):
        sys.path.insert(0, p)
        break

from src.rag import config
from src.rag.data import loader
from src.rag.eval import ragas_eval

try:
    import ragas
except ImportError as e:
    raise ImportError(
        "RAGAS is not installed. Run `pip install -r requirements-ragas.txt` "
        "or `pip install ragas==0.4.3 sentence-transformers`."
    ) from e

os.environ.setdefault("RAGAS_DO_NOT_TRACK", "true")

if "gen_records" not in globals():
    with open(config.RESULTS_DIR / "generation_predictions.json", encoding="utf-8") as f:
        cache = json.load(f)
    gen_records = [v for k, v in cache.items() if k != "__signature__"]

corpus = globals().get("corpus") or {}
if not corpus and any(not r.get("context_texts") for r in gen_records):
    corpus = loader.load_corpus()

ragas_rows = ragas_eval.build_ragas_rows(
    gen_records, corpus, sample_size=config.RAGAS_SAMPLE_SIZE
)
print("RAGAS sample:", len(ragas_rows), "of", len(gen_records))
if ragas_rows:
    sizes = [len(row["retrieved_contexts"]) for row in ragas_rows]
    print(
        "contexts per sample: min", min(sizes),
        "max", max(sizes),
        "avg", sum(sizes) / len(sizes),
    )


In [ ]:
import os, sys
for p in [os.path.abspath(".."), os.path.abspath(".")]:
    if os.path.exists(os.path.join(p, "src", "rag")):
        sys.path.insert(0, p)
        break

from src.rag import config
from src.rag.eval import ragas_eval

assert "ragas_rows" in globals() and ragas_rows, (
    "No RAGAS rows; run the previous cell first (it loads the cached generation records)."
)

ragas_results = await ragas_eval.evaluate_ragas(
    ragas_rows,
    concurrency=config.RAGAS_CONCURRENCY,
    cache_path=config.RESULTS_DIR / "ragas_predictions.json",
)
ragas_summary = ragas_eval.summarize_ragas_results(ragas_results)

for name, stats in ragas_summary["scores"].items():
    mean = "n/a" if stats["mean"] is None else f"{stats['mean']:.4f}"
    print(f"{name:22s} {mean:>8s}  n={stats['n']}  errors={stats['errors']}")

ragas_eval.write_ragas_result(config.RESULTS_DIR / "ragas_result.md", ragas_summary)
print("wrote:", config.RESULTS_DIR / "ragas_result.md")
